In [150]:
import numpy as np
import pandas as pd

In [151]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [152]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [153]:
X = df.drop(columns=["Class"])
y = df["Class"]

In [154]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [155]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [156]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

# ANN

In [168]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [169]:
#Convert data into ==>>>> Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)   # here .values are not needed because output are not continues values

X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [170]:
# Tensor Dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [171]:
# Data Loader
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size= 32)

# Build our Model

In [172]:
X.shape[1]

34

In [173]:
# 
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.model = nn.Sequential(
            # 1st 
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            # 2nd
            nn.Linear(64,64),
            nn.ReLU(),
            # 3rd
            nn.Linear(64,7)
        )
    def forward(self ,x):
        return self.model(x)

In [174]:
model = ANN()
# loss & optimiczer
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [175]:
# Training the NN
epochs = 100
for epoch in range(epochs):
    model.train()
    
    running_loss = 0.0
    
    for xb, yb in train_loader:
        # yb = yb[:, 0]
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update
        
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    print(f"epoch = {epoch}/{epochs}, loss = {train_loss}  ")

epoch = 0/100, loss = 1.7429155163142993  
epoch = 1/100, loss = 1.164492754832558  
epoch = 2/100, loss = 0.7350756862889165  
epoch = 3/100, loss = 0.548864625070406  
epoch = 4/100, loss = 0.44564957916736603  
epoch = 5/100, loss = 0.3848108532636062  
epoch = 6/100, loss = 0.3373697633328645  
epoch = 7/100, loss = 0.31874530470889545  
epoch = 8/100, loss = 0.2885408252477646  
epoch = 9/100, loss = 0.26604437147793564  
epoch = 10/100, loss = 0.2428387776017189  
epoch = 11/100, loss = 0.23890729021766913  
epoch = 12/100, loss = 0.22653924285069757  
epoch = 13/100, loss = 0.21939792726998744  
epoch = 14/100, loss = 0.21978312795576843  
epoch = 15/100, loss = 0.20170514155989108  
epoch = 16/100, loss = 0.1914625627839047  
epoch = 17/100, loss = 0.18306078826603683  
epoch = 18/100, loss = 0.17221203025268472  
epoch = 19/100, loss = 0.1652973828756291  
epoch = 20/100, loss = 0.16525339659141458  
epoch = 21/100, loss = 0.15865736114590065  
epoch = 22/100, loss = 0.1506250

In [176]:
print(yb.shape)

torch.Size([14])


# Evaluate

In [177]:
model.eval() # using this we can put our model into evaluation step
total = 0
correct = 0
with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)   # [0.2, 0.5, 1.3, ...] 7 vals
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual  samples in each batch 

print("Accuracy :-", (correct/total)*100)

Accuracy :- 91.66666666666666
